# Scaling and Distance

**Project question:** Which customer looks nearest when the variables use incompatible units?

By the end of this notebook, you should be able to:

- explain how a large numerical scale can dominate Euclidean distance
- standardize features before comparing distances
- recognize that scaling does not make every feature substantively relevant

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

In [ ]:
df = pd.read_csv(DATA / 'simulated_customer_profiles.csv')
features = ['visits_per_month', 'avg_order_value', 'discount_rate', 'email_opens', 'tenure_months', 'support_contacts', 'returns_per_year']
X = df[features]
X.describe()

In [ ]:
raw_dist = pairwise_distances(X)
scaler = StandardScaler().fit(X)
scaled = scaler.transform(X)
scaled_dist = pairwise_distances(scaled)
np.fill_diagonal(raw_dist, np.inf)
np.fill_diagonal(scaled_dist, np.inf)
query = 0
raw_neighbor = int(np.argmin(raw_dist[query]))
scaled_neighbor = int(np.argmin(scaled_dist[query]))
pd.DataFrame([
    {
        'space': 'raw units',
        'query_customer': df.loc[query, 'customer_id'],
        'nearest_customer': df.loc[raw_neighbor, 'customer_id'],
        'distance': raw_dist[query, raw_neighbor],
    },
    {
        'space': 'standardized',
        'query_customer': df.loc[query, 'customer_id'],
        'nearest_customer': df.loc[scaled_neighbor, 'customer_id'],
        'distance': scaled_dist[query, scaled_neighbor],
    },
])

In [ ]:
scale_table = pd.DataFrame({'feature': features, 'std_dev': X.std().values}).sort_values('std_dev', ascending=False)
scale_table

**Interpretation:** In raw units, features such as average order value or tenure can dominate features measured on smaller scales. Standardization gives each feature variance one in this sample, so the nearest neighbor can change.

Scaling is usually necessary for KNN, K-means, PCA on a correlation scale, and penalized regression. It does not decide whether Euclidean distance is meaningful, whether features deserve equal weight, or whether outliers need robust treatment.